# GP_FSA — module tests

Load CW (financial statement) data from MongoDB via **GP_KDB**, then compute derived ratios with **GP_FSA**.

**How to run**
1. Open this file in Cursor / VS Code / Jupyter
2. Select kernel: `C:\Project\K_VENV\Scripts\python.exe`
3. Change `TEST_IDE` below if you want another stock
4. Run all cells top to bottom

**Requires:** MongoDB running, `pip install -e modules/GP_KDB` and `pip install -e modules/GP_FSA`

In [1]:
import sys
import pandas as pd

sys.path.insert(0, r'C:/Project/modules')

from GP_KDB import KDB
from GP_FSA import Stock, accounts

TEST_IDE = 'sh600519'   # e.g. sz000001, sh689009

## 1. Connect and load financial statements

In [12]:
KDB().setting()
st = Stock(TEST_IDE).load_FS()

print('IDE:', TEST_IDE)
print('FS shape:', st.FS.shape)
print('Registered accounts/ratios:', len(accounts))
st.FS.tail(3)

IDE: sh600519
FS shape: (106, 586)
Registered accounts/ratios: 430


,CODE,col1,col10,col100,col101,col102,col103,col104,col105,col106,...,col91,col92,col93,col94,col95,col96,col97,col98,col99,PKV
REPORTDATE,,,,,,,,,,,,,,,,,,,,,
2025-09-30,600519,51.53,5.209530e+09,4.429718e+09,1.367335e+11,8.595278e+09,1.239348e+10,5.910871e+10,7.567357e+09,9.853672e+10,...,0.0,8.940344e+10,2.250464e+10,0.0,6.689880e+10,6.462675e+10,2.272058e+09,1.387917e+11,0.0,5.116151
2025-12-31,600519,65.66,0.000000e+00,4.712295e+09,1.861288e+11,1.145014e+10,1.547346e+10,7.743060e+10,9.012158e+09,1.246066e+11,...,0.0,1.147553e+11,2.944494e+10,0.0,8.531032e+10,8.232006e+10,2.990258e+09,1.839904e+11,0.0,5.116151
2026-03-31,600519,21.76,0.000000e+00,4.154339e+08,5.618097e+10,3.211262e+09,7.351815e+09,2.403097e+10,1.463836e+09,2.927107e+10,...,0.0,3.754325e+10,9.389418e+09,0.0,2.815383e+10,2.724251e+10,9.113186e+08,5.639259e+10,0.0,5.116151


## 2. Raw ratios from CW (MongoDB col197 etc.)

In [3]:
roe = accounts['净资产收益率'].value(st)
eps = accounts['基本每股收益'].value(st)

pd.DataFrame({'ROE': roe, 'EPS': eps}).dropna().tail(8)

,ROE,EPS
REPORTDATE,,
2024-06-30,19.076,33.19
2024-09-30,25.589,48.42
2024-12-31,36.991,68.64
2025-03-31,10.392,21.38
2025-06-30,19.025,36.18
2025-09-30,25.140,51.53
2025-12-31,33.650,65.66
2026-03-31,10.057,21.76


## 3. Calculated ratios (GP_FSA formulas)

In [4]:
bps = accounts['BPS'].value(st)
roe_1y = accounts['净资产收益率_1Y'].value(st)
gross_margin = accounts['毛利率'].value(st)

pd.DataFrame({
    'BPS': bps,
    'ROE_1Y': roe_1y,
    '毛利率': gross_margin,
}).dropna().tail(8)

,BPS,ROE_1Y,毛利率
REPORTDATE,,,
2024-06-30,682.534577,0.383544,0.702432
2024-09-30,690.968187,0.363051,0.695475
2024-12-31,689.002829,0.384266,0.700346
2025-03-31,699.339408,0.357413,0.731935
2025-06-30,691.363589,0.393439,0.702197
2025-09-30,699.151887,0.363930,0.696669
2025-12-31,694.170197,0.344637,0.679994
2026-03-31,704.415982,0.312599,0.696300


## 4. Free cash flow (calced_fcf)

In [5]:
fcf = accounts['自由现金流'].value(st)
fcf.dropna().tail(8)

REPORTDATE
2024-06-30    1.007112e+11
2024-09-30    1.317640e+11
2024-12-31    1.954084e+11
2025-03-31    5.288094e+10
2025-06-30    1.068105e+11
2025-09-30    1.464517e+11
2025-12-31    1.878647e+11
2026-03-31    5.756035e+10
dtype: float64

## 5. Main FSA dashboard (`main_FSA`)

In [6]:
dashboard = st.main_FSA()
print('Columns:', len(dashboard.columns))
dashboard.tail(5)

Columns: 40


,REPORTDATE,BPS,BPS_low,BPS_low_low,Op_Revenue_PS,Op_Cost_PS,Op_fixed_Cost_PS,Op_fixed_Cost_without_DA_PS,BPS_and_Op_NetCashflow_PS,cashin_OP_PS,...,C_负债合计,C_负债合计_dY,C_流动负债合计,C_流动负债合计_dY,C_少数股东权益,C_每股流动资产合计,C_FQ每股净流动资产,C_EBIT,C_EBITDA,DTnum
101,2025-03-31,699.339408,695.863824,695.863824,660.604246,409.671221,547.006420,543.238596,729.976917,669.784306,...,4.417836e+10,7.190589e+09,4.387018e+10,7.243506e+09,9.832492e+09,702.460294,684.384626,800.621533,801.739223,12521
102,2025-06-30,691.363589,687.624105,687.624105,662.098885,412.049304,548.939078,545.078955,716.874394,668.790849,...,4.312220e+10,-8.040247e+09,4.284285e+10,-7.964029e+09,1.048905e+10,694.200216,674.924481,798.103959,799.299860,12612
103,2025-09-30,699.151887,695.712358,695.712358,662.537030,411.883182,548.701859,544.986408,728.224574,669.690815,...,3.903316e+10,-4.939776e+07,3.876338e+10,3.948954e+07,8.635060e+09,698.962968,682.503011,801.240583,802.354912,12704
104,2025-12-31,694.170197,690.537194,690.474485,656.901235,412.879451,549.535778,545.611506,716.715086,665.538466,...,4.987559e+10,-7.057674e+09,4.961048e+10,-6.905512e+09,9.321443e+09,697.356660,675.374319,794.616189,795.881555,12796
105,2026-03-31,704.415982,701.169423,701.112978,658.851421,422.274264,553.940597,550.187791,730.312739,665.674126,...,3.878296e+10,-5.395403e+09,3.845560e+10,-5.414584e+09,1.024185e+10,704.649616,689.301562,798.756377,799.970980,12886


## 6. Browse available accounts

In [7]:
sorted(accounts.keys())[:30], '...', len(accounts)

(['BPS',
  'BPS_low',
  'BPS_low_low',
  'BPS加5年净利润',
  'BPS加5年自由现金流',
  'BPS去除商誉',
  'DA',
  'EBIT',
  'EBITDA',
  'EBITDA_营业总收入TpctT非金融',
  'EBITDA_负债合计TpctT非金融',
  'EPS',
  'FQ每股净流动资产',
  'FQ每股流动资产',
  'Financing_Obli',
  'Gearing Ratio',
  'LT_Debt',
  'Net_Financing_Obli',
  'Op_Cost',
  'Op_EBITDA_revenue',
  'Op_NetCashflow',
  'Op_Revenue',
  'Op_fixed_Cost',
  'Op_fixed_Cost_without_DA',
  'PKV',
  'QFII持股量',
  'QFII机构数',
  'ST_Debt',
  'YEBIT',
  'YEBITDA'],
 '...',
 430)

## 7. Ratio table helper (from CW directly)

In [ ]:
# from GP_FSA.statement.Ratio import ratioTable
from GP_FSA.statement.BS import LIAB
from GP_FSA.table.balanceSheet import tbl_ASSET
tbl_ASSET.get_df(st).iloc[:15, :5]
# tbl_ASSET(TEST_IDE).iloc[:8, -5:]
# ratioTable(TEST_IDE).iloc[:8, -5:]

,propert,2026-03-31,2025-12-31,2025-09-30,2025-06-30
货币资金,"{'taglevel': 1, 'color': 'blue', 'isPercent': ...",48786690048.0,51690610688.0,51753058304.0,51644878848.0
期末现金及现金等价物余额,"{'taglevel': 2, 'color': 'orchid', 'tag': '其中：...",177968234496.0,126425612288.0,159502467072.0,142885257216.0
不属于现金及现金等价物的货币资金,"{'taglevel': 2, 'color': 'orchid', 'tag': '其中：...",-129181544448.0,-74735001600.0,-107749408768.0,-91240378368.0
货币资金比率T非金融,"{'taglevel': 2, 'color': 'green', 'tag': '货币资金...",17.968,20.47,20.17,21.038
结算备付金,"{'taglevel': 1, 'color': 'blue', 'isPercent': ...",0.0,0.0,0.0,0.0
拆出资金,"{'taglevel': 1, 'color': 'blue', 'isPercent': ...",155995390000.0,99096190000.0,135402540000.0,126350830000.0
发放贷款及垫款_CA,"{'taglevel': 1, 'color': 'blue', 'isPercent': ...",0.0,0.0,0.0,0.0
交易性金融资产,"{'taglevel': 1, 'color': 'blue', 'isPercent': ...",0.0,0.0,0.0,0.0
衍生金融资产,"{'taglevel': 1, 'color': 'blue', 'isPercent': ...",0.0,0.0,0.0,0.0
应收票据及应收账款_CA,"{'taglevel': 1, 'color': 'blue', 'isPercent': ...",32344588.0,2609048.5,5235061760.0,2853742336.0


## 8. Try any account by name

In [26]:
ACCOUNT_NAME = 'BPS'   # change to any key in accounts

if ACCOUNT_NAME in accounts:
    print(accounts[ACCOUNT_NAME].value(st).dropna().tail(8))
else:
    print('Not found. Sample names:', sorted(accounts.keys())[:20])


REPORTDATE
2024-06-30    682.534577
2024-09-30    690.968187
2024-12-31    689.002829
2025-03-31    699.339408
2025-06-30    691.363589
2025-09-30    699.151887
2025-12-31    694.170197
2026-03-31    704.415982
dtype: float64
